In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
class Expert(nn.Module):
    def __init__(self, d_model, dim_feedforward, dropout):
        super().__init__()
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        self.ffn(x)

In [3]:
class TopKRouter(nn.Module):
    def __init__(self, d_model, num_experts, top_k=2):
        super().__init__()
        self.num_experts = self.num_experts
        self.top_k = top_k
        self.gate = nn.Linear(d_model, num_experts)
        
    def forward(self, x):
        # Calculate routing scores
        logits = self.gate(x)

        # Get top-k experts
        top_k_logits, top_k_indices = torch.topk(logits, self.top_k, dim=-1)
        
        # Softmax over top-k
        top_k_probs = F.softmax(top_k_indices, dim=-1)
        
        return top_k_probs, top_k_indices

In [ ]:
class MoELayer(nn.Module):
    def __init__(self, d_model, num_exparts=8, top_k=2, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.num_exparts = num_exparts
        self.top_k = top_k
        
        self.exparts = nn.ModuleList([
            Expert(d_model, dim_feedforward, dropout)
            for _ in range(num_exparts)
        ])
        
        self.router = TopKRouter(d_model, num_exparts, top_k)
    
    
    def forward(self, x):
        batch_size, seq_len, d_model = x.shape
        
        probs, indices = self.router(x)
        
        x_flat = x.view(-1, d_model)
        probs_flat = probs.view(-1, self.top_k)
        indices_flat = indices.view(-1, self.top_k)
        
        output = torch.zeros_like(x_flat)
        
        for expart_idx in range(self.num_exparts):
            expert_mask = (indices_flat == expart_idx).any(dim=1)
            
            if expert_mask.any():
                expert_input = x_flat[expert_mask]
                
                expert_output = self.exparts[expart_idx](expert_input)
                
                expert_probs = probs_flat[expert_mask]
                expert_weights = expert_probs[:, (indices_flat[expert_mask] == expart_idx).nonzero(as_tuple=True)[1]]
                
                output[expert_mask] += expert_output * expert_weights.unsqueeze(-1)
        
        return output.view(batch_size, seq_len, d_model)

In [ ]:
class MoETransformerBlock(nn.Module):
    def __init__(self, d_model, num_experts=8, top_k=2):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, num_heads=8)
        self.moe = MoELayer(d_model, num_experts, top_k)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x):
        x = x + self.self_attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.moe(self.norm2(x))
        return x

**Load Balancing**

In [ ]:
def load_balancing_loss(router_probs, expert_indices, num_experts):
    # Calculate fraction of tokens routed to each expert
    expert_usage = torch.zeros(num_experts, device=router_probs.device)
    
    for i in range(num_experts):
        mask = (expert_indices == i).any(dim=-1)
        expert_usage[i] = mask.float().mean()
    
    # Load balancing loss (encourage uniform distribution)
    uniform_dist = torch.ones(num_experts, device=router_probs.device) / num_experts
    lb_loss = torch.nn.functional.mse_loss(expert_usage, uniform_dist)
    
    return lb_loss


# Add to training loss
total_loss = model_loss + 0.01 * load_balancing_loss(probs, indices, num_experts)